# STEP 1: Anthropic API 最初のリクエスト

このノートブックでは、Claude API への基本的なリクエストを実践します。

## 1. 必要なパッケージのインストール

In [2]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. 環境変数の読み込みとクライアントの作成

In [3]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

print("クライアントの作成に成功しました")

クライアントの作成に成功しました


## 3. 最初のAPIリクエスト

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "量子コンピューティングとは何ですか？一文で答えてください"
        }
    ]
)

print(message)

## 4. レスポンスからテキストを抽出

In [5]:
print(message.content[0].text)

Quantum computing is a type of computing that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain complex problems much faster than classical computers.


---

# STEP 2: 複数ターンの会話（会話履歴の管理）

Claude はリクエスト間で会話を記憶しません。
文脈を維持するには、**会話履歴をこちら側で管理**してリクエストに含める必要があります。

## 5. ヘルパー関数の定義

In [6]:
def add_user_message(messages, text):
    """ユーザーメッセージを履歴に追加する"""
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    """アシスタントメッセージを履歴に追加する"""
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    """会話履歴全体をClaudeに送信してレスポンスを返す"""
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

print("ヘルパー関数の定義が完了しました")

ヘルパー関数の定義が完了しました


## 6. 複数ターンの会話を実行する

In [ ]:
# 空のメッセージリストからスタート
messages = []

# 最初の質問を追加
add_user_message(messages, "量子コンピューティングを一文で定義してください")

# Claudeの回答を取得
answer = chat(messages)
print(f"Claude (1回目): {answer}")

# Claudeの回答を履歴に追加
add_assistant_message(messages, answer)

# フォローアップの質問を追加
add_user_message(messages, "もう一文書いてください")

# 会話履歴全体を含めてリクエスト
final_answer = chat(messages)
print(f"Claude (2回目): {final_answer}")

## 7. 送信している会話履歴を確認する

In [8]:
# 2回目のリクエスト時点でのメッセージ履歴を確認
import json
print(json.dumps(messages, indent=2, ensure_ascii=False))

[
  {
    "role": "user",
    "content": "Define quantum computing in one sentence"
  },
  {
    "role": "assistant",
    "content": "Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers."
  },
  {
    "role": "user",
    "content": "Write another sentence"
  }
]


---

# STEP 3: システムプロンプト

システムプロンプトを使うと、Claude の振る舞い・トーン・役割をカスタマイズできます。
ユーザーの質問とは別に、会話全体を通じて有効な「指示」として機能します。

## 8. chat関数をシステムプロンプト対応に更新する

Claude の API は `system=None` を受け付けないため、指定がある場合だけ含める実装にします。

In [9]:
def chat(messages, system=None):
    """会話履歴全体をClaudeに送信してレスポンスを返す（システムプロンプト対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    # system=None の場合はパラメータに含めない
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 9. システムプロンプトなしで質問する（通常の回答）

In [ ]:
messages = []
add_user_message(messages, "5x + 2 = 3 はどうやって解きますか？")

answer = chat(messages)
print("【システムプロンプトなし】")
print(answer)

## 10. システムプロンプトあり（数学家庭教師として振る舞う）

In [ ]:
system_prompt = """
あなたは忍耐強い数学の家庭教師です。
生徒の質問に直接答えないでください。
ステップバイステップで解決策へ導いてください。
"""

messages = []
add_user_message(messages, "5x + 2 = 3 はどうやって解きますか？")

answer = chat(messages, system=system_prompt)
print("【システムプロンプトあり（数学家庭教師）】")
print(answer)

---

# STEP 4: Temperature（温度）

`temperature` は Claude の回答の「ランダム性・創造性」を制御するパラメータです。

| 温度 | 特性 | 用途例 |
|------|------|--------|
| 0.0〜0.3（低） | 決定論的・一貫性が高い | コード生成、データ抽出、事実回答 |
| 0.4〜0.7（中） | バランス型 | 要約、教育コンテンツ、問題解決 |
| 0.8〜1.0（高） | 多様・創造的 | ブレインストーミング、創作、マーケティング |

## 11. chat関数にtemperatureパラメータを追加する

In [12]:
def chat(messages, system=None, temperature=1.0):
    """会話履歴全体をClaudeに送信してレスポンスを返す（temperature対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 12. 低温（0.0）vs 高温（1.0）を比較する

同じプロンプトを温度違いで3回ずつ実行して、回答のばらつきを確認します。

In [ ]:
prompt = "タイムトラベルをテーマにした映画のアイデアを一文で教えてください。"

print("=== 低温 (temperature=0.0) ===")
for i in range(3):
    messages = []
    add_user_message(messages, prompt)
    answer = chat(messages, temperature=0.0)
    print(f"  {i+1}回目: {answer}")

print()
print("=== 高温 (temperature=1.0) ===")
for i in range(3):
    messages = []
    add_user_message(messages, prompt)
    answer = chat(messages, temperature=1.0)
    print(f"  {i+1}回目: {answer}")

---

# STEP 5: ストリーミング

通常のリクエストは応答が完成するまで待つのに対し、ストリーミングはテキストが生成されるたびにチャンクごとに受信できます。
チャットアプリのような「文字が流れてくる」UXを実現するための機能です。

## 13. ストリームイベントの中身を確認する

`stream=True` を指定すると、どんなイベントが送られてくるかを見てみます。

In [ ]:
messages = []
add_user_message(messages, "架空のデータベースを一文で説明してください")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

## 14. テキストだけをリアルタイムで表示する

SDKの簡略インターフェース `messages.stream` を使うと、テキスト以外のイベントを自動でフィルタリングできます。
`end=""` により改行なしで文字が流れるように表示されます。

In [ ]:
messages = []
add_user_message(messages, "架空のデータベースを一文で説明してください")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

## 15. ストリーミングしながら完全なメッセージも取得する

リアルタイム表示と同時に、DB保存や後続処理用の完全なメッセージオブジェクトも取得できます。

In [ ]:
messages = []
add_user_message(messages, "架空のデータベースを一文で説明してください")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    # リアルタイムでテキストを表示
    for text in stream.text_stream:
        print(text, end="", flush=True)

    # ストリーミング完了後に完全なメッセージを取得
    final_message = stream.get_final_message()

print("\n\n--- final_message ---")
print(f"stop_reason : {final_message.stop_reason}")
print(f"input_tokens: {final_message.usage.input_tokens}")
print(f"output_tokens: {final_message.usage.output_tokens}")

---

# STEP 6: 構造化データの出力制御（プリフィル + 停止シーケンス）

ClaudeはデフォルトでJSONをマークダウンコードブロックや説明文で囲んで返します。
**アシスタントメッセージのプリフィル**と**停止シーケンス**を組み合わせることで、生データだけを取得できます。

## 16. デフォルト（プリフィルなし）の出力を確認する

In [ ]:
messages = []
add_user_message(messages, "EventBridgeルールをJSONで生成してください（短くてOK）")

text = chat(messages)
print(text)

## 17. chat関数に stop_sequences パラメータを追加する

In [18]:
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """会話履歴全体をClaudeに送信してレスポンスを返す（stop_sequences対応）"""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

print("chat関数の更新が完了しました")

chat関数の更新が完了しました


## 18. プリフィル + 停止シーケンスで生JSONだけを取得する

- `add_assistant_message(messages, "```json")` でClaudeに「コードブロックはもう始まっている」と思わせる
- `stop_sequences=["` ``` `"]` でClaudeが閉じタグを書こうとした瞬間に停止させる

In [ ]:
import json

messages = []
add_user_message(messages, "EventBridgeルールをJSONで生成してください（短くてOK）")
add_assistant_message(messages, "```json")  # プリフィル：コードブロックが既に始まっているように見せる

text = chat(messages, stop_sequences=["```"])  # 閉じタグが来たら停止

print("--- 生のレスポンス ---")
print(repr(text))

print("\n--- パース済みJSON ---")
clean_json = json.loads(text.strip())
print(json.dumps(clean_json, indent=2))

---

# STEP 7: プロンプト評価ワークフロー

プロンプトを「なんとなくテスト」するのではなく、スコアで客観的に測定・比較する仕組みを作ります。

**5ステップの流れ：**
1. プロンプトを作成する
2. 評価データセットを用意する
3. 各質問をClaudeに送って回答を得る
4. 採点者（Claude）にスコアをつけさせる
5. プロンプトを改善して繰り返す

## 19. STEP1〜3: プロンプトと評価データセットを用意し、Claudeに回答させる

In [ ]:
# STEP1: 評価対象のプロンプト（シンプルな初期バージョン）
def build_prompt(question):
    return f"""ユーザーの質問に答えてください:

{question}"""

# STEP2: 評価データセット
questions = [
    "2+2はいくつですか？",
    "オートミールの作り方を教えてください",
    "月までの距離はどれくらいですか？",
]

# STEP3: 各質問をClaudeに送って回答を収集
answers = []
for question in questions:
    messages = []
    add_user_message(messages, build_prompt(question))
    answer = chat(messages, temperature=0.0)  # 再現性のため低温に設定
    answers.append(answer)
    print(f"Q: {question}")
    print(f"A: {answer[:100]}...")  # 長い場合は先頭100文字だけ表示
    print()

## 20. STEP4: 採点者（Claude）にスコアをつけさせる

In [ ]:
def grade_answer(question, answer):
    """Claudeに回答を1〜10で採点させる"""
    grader_prompt = f"""あなたは客観的な採点者です。以下の回答を1〜10で採点してください。
10 = 完璧な回答、1 = 完全に間違いまたは役に立たない回答。
1から10の整数のみを返してください。それ以外のテキストは含めないでください。

質問: {question}
回答: {answer}"""

    messages = []
    add_user_message(messages, grader_prompt)
    score_text = chat(messages, temperature=0.0)
    return int(score_text.strip())

# 各回答をスコアリング
scores = []
for question, answer in zip(questions, answers):
    score = grade_answer(question, answer)
    scores.append(score)
    print(f"Q: {question}")
    print(f"スコア: {score}/10")
    print()

avg_score = sum(scores) / len(scores)
print(f"平均スコア: {avg_score:.2f}/10")

## 21. STEP5: プロンプトを改善して再評価する

プロンプトに「詳しく答えてください」を追加して、スコアが変わるか比較します。

In [ ]:
# 改善版プロンプト
def build_prompt_v2(question):
    return f"""ユーザーの質問に答えてください:

{question}

十分な詳細を含めて答えてください。"""

# 改善版で回答を収集してスコアリング
scores_v2 = []
for question in questions:
    messages = []
    add_user_message(messages, build_prompt_v2(question))
    answer = chat(messages, temperature=0.0)
    score = grade_answer(question, answer)
    scores_v2.append(score)
    print(f"Q: {question}")
    print(f"スコア: {score}/10")
    print()

avg_score_v2 = sum(scores_v2) / len(scores_v2)

print(f"初期プロンプト 平均スコア: {avg_score:.2f}/10")
print(f"改善プロンプト 平均スコア: {avg_score_v2:.2f}/10")
print(f"差分: {avg_score_v2 - avg_score:+.2f}")